# D7_crying — 무라벨(라벨링 없이) 획득

AudioSet 527클래스로 사전학습된 AST 체크포인트(MIT/ast-finetuned-audioset-10-10-0.4593)를 제로샷으로 사용해 클립 오디오(AAC 8kHz→16kHz 리샘플)의 "Baby cry, infant cry" 클래스 시그모이드 확률을 계산한다. AST는 AudioSet mAP 0.485로 힌트의 PANNs CNN14(0.431)보다 정확하고 HF transformers 한 줄로 로드되어 T4 1개에서 충분히 빠르다. 보조로 MediaPipe Face Mesh를 manifest의 sound_iv 소리 구간 중앙 프레임에 적용해 울음 표정 기하(입벌림 MAR, 눈뜸 EAR)를 측정하고, 오디오 확률이 경계(0.2~0.5)인 클립은 입벌림 동반 시에만 양성으로 판정하는 규칙으로 의사라벨을 만든다. 수동 라벨은 전혀 사용하지 않는다.

In [ ]:
# 설치 셀 — 처음 실행 후 protobuf/numpy 경고가 나오면: 런타임 → 세션 다시 시작 → 이어서 실행
!pip install -q transformers mediapipe==0.10.14 av decord
!pip -q uninstall -y tensorflow  # mediapipe 0.10.14(protobuf4)와 TF(protobuf5) 충돌 방지 — 이 노트북은 TF 미사용

In [ ]:
# 공통 설정 — Drive 마운트 + 경로 + manifest
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import pandas as pd

BASE = Path('/content/drive/MyDrive/BabyMon/dataset')   # ← 업로드 위치
CLIPS = None
for cand in (BASE/'clips', BASE/'resized_2', BASE):
    if cand.is_dir() and next(cand.glob('*.mp4'), None):
        CLIPS = cand; break
assert CLIPS, 'mp4 폴더 없음 (clips/ 또는 resized_2/)'
assert (BASE/'manifest.csv').exists(), 'manifest.csv 를 BASE 에 업로드하세요'
WORK = Path('/content/work'); WORK.mkdir(exist_ok=True)
man = pd.read_csv(BASE/'manifest.csv')
print('clips:', CLIPS, len(list(CLIPS.glob("*.mp4"))), '개 / manifest:', len(man))

## D7_crying — 울음 자동 라벨 (오디오 제로샷 + 얼굴 기하 보조)

**방법**: AudioSet 527클래스로 사전학습된 **AST**(`MIT/ast-finetuned-audioset-10-10-0.4593`)를 제로샷으로 사용해 클립별 **"Baby cry, infant cry" 클래스 확률**을 계산한다 (수동 라벨 불필요). AST는 AudioSet mAP 0.485로 PANNs CNN14(0.431)보다 높고 transformers 한 줄로 로드된다.

보조 신호로 **MediaPipe Face Mesh**(468 랜드마크)를 manifest의 `sound_iv` 소리 구간 중앙 프레임에 적용해 울음 표정(입 크게 벌림 MAR ↑, 눈 감음 EAR ↓)을 측정하고, 오디오 확률이 경계인 클립의 의사라벨을 보정한다.

- 오디오: AAC 8kHz mono → PyAV 디코드 후 16kHz 리샘플(AST 입력 규격). 울음 기본주파수(~300–600Hz)와 주요 하모닉은 4kHz 이하라 8kHz 원본으로도 유효.
- 의사라벨 규칙: `cry_prob ≥ 0.5` → 울음(1), `0.2 ≤ cry_prob < 0.5` 이고 입벌림 `MAR ≥ 0.45` 동반 → 울음(1), 그 외 0. 임계값은 아래 검증 셀의 분포를 보고 조정.

**출처**
- AST: Audio Spectrogram Transformer (Interspeech 2021) — https://arxiv.org/abs/2104.01778
- PANNs: Large-Scale Pretrained Audio Neural Networks (arXiv 2019, IEEE TASLP 2020) — https://arxiv.org/abs/1912.10211
- MediaPipe Face Mesh — https://mediapipe.readthedocs.io/en/latest/solutions/face_mesh.html

In [ ]:
import torch, numpy as np
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

device = 'cuda' if torch.cuda.is_available() else 'cpu'
AST_CKPT = 'MIT/ast-finetuned-audioset-10-10-0.4593'
fe = AutoFeatureExtractor.from_pretrained(AST_CKPT)
ast = AutoModelForAudioClassification.from_pretrained(AST_CKPT).to(device).eval()

# AudioSet 라벨 중 울음 관련 클래스 인덱스
id2label = ast.config.id2label
BABY_ID = [i for i, l in id2label.items() if 'baby cry' in l.lower()][0]
CRY_IDS = [i for i, l in id2label.items()
           if any(k in l.lower() for k in ['baby cry', 'crying', 'whimper', 'wail'])]
print('baby cry id:', BABY_ID, '| 관련 클래스:', {i: id2label[i] for i in CRY_IDS})

# MediaPipe Face Mesh (보조 시각 신호) — 실패 시 오디오 단독 모드로 동작
try:
    import mediapipe as mp
    mp_face = mp.solutions.face_mesh.FaceMesh(
        static_image_mode=True, max_num_faces=1,
        refine_landmarks=False, min_detection_confidence=0.3)
except Exception as e:  # mediapipe 버전에 따라 solutions API 변경 가능성 대비
    mp_face = None
    print('MediaPipe 사용 불가, 오디오 단독 모드:', e)

In [ ]:
import av

SR = 16000  # AST 입력 샘플레이트

def load_audio(path, sr=SR):
    """mp4의 AAC 오디오를 mono float32 16kHz로 디코드 (PyAV)."""
    buf = []
    try:
        with av.open(str(path)) as c:
            rs = av.AudioResampler(format='s16', layout='mono', rate=sr)
            for fr in c.decode(audio=0):
                out = rs.resample(fr)
                if not isinstance(out, list):  # PyAV<9 는 단일 프레임 반환
                    out = [out]
                buf += [f.to_ndarray().reshape(-1) for f in out if f is not None]
    except Exception:
        return None
    if not buf:
        return None
    return np.concatenate(buf).astype(np.float32) / 32768.0

@torch.no_grad()
def audio_cry(wav):
    """AST 제로샷: (baby cry 확률, 울음 관련 클래스 최대 확률) 반환."""
    ins = fe(wav, sampling_rate=SR, return_tensors='pt')
    p = torch.sigmoid(ast(ins['input_values'].to(device)).logits[0]).cpu().numpy()
    return float(p[BABY_ID]), float(p[CRY_IDS].max())

In [ ]:
import json
from decord import VideoReader

def face_feats(path, ivs, n=5):
    """소리 구간 중앙 프레임에서 입벌림(MAR)/눈뜸(EAR) 측정. 실패해도 안전."""
    out = {'face_rate': 0.0, 'mouth_open_max': 0.0, 'eye_open_min': 1.0}
    if mp_face is None:
        return out
    try:
        vr = VideoReader(str(path))
        fps = vr.get_avg_fps() or 15.0
        dur = (len(vr) - 1) / fps
        ts = [min((a + b) / 2, dur) for a, b in ivs[:n]] if ivs \
            else list(np.linspace(0.2, max(dur, 0.3), n))
        hits = 0
        for t in ts:
            img = vr[min(int(t * fps), len(vr) - 1)].asnumpy()  # RGB uint8
            res = mp_face.process(img)
            if not res.multi_face_landmarks:
                continue
            lm = res.multi_face_landmarks[0].landmark
            px = lambda i: np.array([lm[i].x * 240, lm[i].y * 136])
            dd = lambda a, b: float(np.linalg.norm(px(a) - px(b)))
            mar = dd(13, 14) / max(dd(61, 291), 1e-6)    # 입: 안쪽 위/아래 ÷ 양끝
            ear = dd(159, 145) / max(dd(33, 133), 1e-6)  # 왼눈: 위/아래 ÷ 양끝
            hits += 1
            out['mouth_open_max'] = max(out['mouth_open_max'], round(mar, 3))
            out['eye_open_min'] = min(out['eye_open_min'], round(ear, 3))
        out['face_rate'] = round(hits / max(len(ts), 1), 2)
    except Exception:
        pass
    return out

T_HI, T_LO, MAR_TH = 0.5, 0.20, 0.45
COLS = ['file', 'cry_prob', 'cry_any_prob', 'face_rate', 'mouth_open_max',
        'eye_open_min', 'label_crying', 'error']

def process_clip(row):
    """클립 1개 → dict (오디오 제로샷 + 얼굴 기하 + 의사라벨)."""
    out = {c: 0 for c in COLS}
    out.update(file=row.file, error='')
    wav = load_audio(CLIPS / row.file)
    if wav is not None and len(wav) > SR // 4:
        out['cry_prob'], out['cry_any_prob'] = audio_cry(wav)
    try:
        ivs = json.loads(row.sound_iv) if isinstance(row.sound_iv, str) else []
    except Exception:
        ivs = []
    out.update(face_feats(CLIPS / row.file, ivs))
    out['label_crying'] = int(out['cry_prob'] >= T_HI or
                              (out['cry_prob'] >= T_LO and out['mouth_open_max'] >= MAR_TH))
    return out

In [ ]:
import time
import pandas as pd

OUT = BASE / 'auto_D7_crying.csv'
N_CLIPS = 1000

done = set()
if OUT.exists():
    done = set(pd.read_csv(OUT)['file'].astype(str))

cand = man.sample(n=min(N_CLIPS, len(man)), random_state=42)
# 참고: 양성 위주 표본이 필요하면 man[man.sound_frac > 0.05] 에서 추출
todo = [r for r in cand.itertuples(index=False) if str(r.file) not in done]
print(f'표본 {len(cand)}개 중 신규 {len(todo)}개 처리 (기존 {len(done)}개 건너뜀)')

rows, t0 = [], time.time()
for i, r in enumerate(todo, 1):
    try:
        d = process_clip(r)
    except Exception as e:
        d = {c: np.nan for c in COLS}
        d.update(file=r.file, error=str(e)[:100])
    rows.append(d)
    if i % 50 == 0:
        pd.DataFrame(rows).reindex(columns=COLS).to_csv(
            OUT, mode='a', header=not OUT.exists(), index=False)
        rows = []
        print(f'{i}/{len(todo)} 완료 — {time.time() - t0:.0f}초 경과, '
              f'{(time.time() - t0) / i:.2f}초/클립')
if rows:
    pd.DataFrame(rows).reindex(columns=COLS).to_csv(
        OUT, mode='a', header=not OUT.exists(), index=False)
print('저장 완료:', OUT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from decord import VideoReader

res = pd.read_csv(OUT).drop_duplicates('file')
print(f"총 {len(res)}개 | 울음 의사라벨 {int(res.label_crying.sum())}개 "
      f"({res.label_crying.mean() * 100:.1f}%) | 얼굴 검출률 평균 {res.face_rate.mean():.2f}")

BLUE, INK = '#3b6fb6', '#555555'
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(res.cry_prob.dropna(), bins=40, color=BLUE)
ax[0].set_yscale('log'); ax[0].set_title('AST baby-cry 확률 분포 (log y)', color=INK)
ax[1].scatter(res.cry_prob, res.mouth_open_max, s=8, color=BLUE, alpha=0.4)
ax[1].axvline(T_HI, lw=1, ls='--', color=INK)
ax[1].set_xlabel('cry_prob'); ax[1].set_ylabel('mouth_open_max (MAR)')
ax[1].set_title('오디오 확률 vs 입벌림', color=INK)
for a in ax:
    a.grid(alpha=0.2)
    a.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

# 극단 사례: 확률 상위 5개 + 경계(0.2~0.5) 3개
show = ['file', 'cry_prob', 'mouth_open_max', 'eye_open_min', 'label_crying']
top = res.sort_values('cry_prob', ascending=False).head(5)
mid = res[(res.cry_prob >= T_LO) & (res.cry_prob < T_HI)].head(3)
display(top[show]); display(mid[show])

fig, ax = plt.subplots(1, 3, figsize=(9, 2.2))
for a, fn in zip(ax, list(top.file[:3])):
    try:
        vr = VideoReader(str(CLIPS / fn))
        a.imshow(vr[len(vr) // 2].asnumpy())
        a.set_title(str(fn)[:24], fontsize=7, color=INK)
    except Exception:
        pass
    a.axis('off')
plt.suptitle('cry_prob 상위 클립 중앙 프레임', fontsize=9, color=INK)
plt.tight_layout(); plt.show()